Data Preprocessing and Aggregation

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta, datetime

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change working directory to project folder
try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")

Mounted at /content/drive
Directory changed


In [ ]:
df_original = pd.read_csv('data/food_waste_augmented.csv')
print(f"Original data shape: {df_original.shape}")

Original data shape: (76290, 27)


In [4]:
df_original.head()

,Date,Meal,Canteen_Section,Food_Category,Waste_Weight_kg,Unit_Price_per_kg,Is_Event_Day,Year,Month,Day,...,Month_Name,Weekday_Type,Cost_Loss,Waste_per_Price,Log_Waste,Foot_Traffic,Is_Waste_Outlier,Is_Cost_Outlier,Is_High_Waste,Is_High_Cost
0,2015-01-01 07:30:00,Breakfast,B,Soup,2.17,1.5,0,2015,1,1,...,January,Weekday,3.25,1.443506,0.772540,96.0,0,0,0,0
1,2015-01-01 06:00:00,Breakfast,C,Meat,2.31,8.0,0,2015,1,1,...,January,Weekday,18.52,0.289299,0.839145,64.0,0,0,0,0
2,2015-01-01 07:00:00,Breakfast,B,Meat,5.00,8.0,0,2015,1,1,...,January,Weekday,40.00,0.625000,1.609438,96.0,0,1,1,1
3,2015-01-01 06:30:00,Breakfast,D,Meat,3.06,8.0,0,2015,1,1,...,January,Weekday,24.46,0.382166,1.117542,120.0,0,1,0,1
4,2015-01-01 08:00:00,Breakfast,D,Rice,5.00,2.0,0,2015,1,1,...,January,Weekday,10.00,2.500000,1.609438,120.0,0,0,1,0


## 1. Parse Timestamps & Create 30-Minute Bins

In [5]:
df = df_original.copy()

# Parse Date column to datetime
df['datetime'] = pd.to_datetime(df['Date'])

# Floor to nearest 30-minute bin
df['time_bin'] = df['datetime'].dt.floor('30min')

print(f"Date range: {df['time_bin'].min()} → {df['time_bin'].max()}")
print(f"Sample time bins:\n{df[['Date', 'datetime', 'time_bin']].head(10)}")

Date range: 2015-01-01 06:00:00 → 2025-08-10 19:30:00
Sample time bins:
                  Date            datetime            time_bin
0  2015-01-01 07:30:00 2015-01-01 07:30:00 2015-01-01 07:30:00
1  2015-01-01 06:00:00 2015-01-01 06:00:00 2015-01-01 06:00:00
2  2015-01-01 07:00:00 2015-01-01 07:00:00 2015-01-01 07:00:00
3  2015-01-01 06:30:00 2015-01-01 06:30:00 2015-01-01 06:30:00
4  2015-01-01 08:00:00 2015-01-01 08:00:00 2015-01-01 08:00:00
5  2015-01-01 08:30:00 2015-01-01 08:30:00 2015-01-01 08:30:00
6  2015-01-01 12:00:00 2015-01-01 12:00:00 2015-01-01 12:00:00
7  2015-01-01 12:30:00 2015-01-01 12:30:00 2015-01-01 12:30:00
8  2015-01-01 12:00:00 2015-01-01 12:00:00 2015-01-01 12:00:00
9  2015-01-01 11:00:00 2015-01-01 11:00:00 2015-01-01 11:00:00


## 2. Aggregate per (Canteen Section, Time Bin)

In [6]:
agg_dict = {
    'Waste_Weight_kg': 'sum',   # total waste per bin
    'Cost_Loss': 'sum',         # total cost loss per bin
    'Foot_Traffic': 'mean'      # average foot traffic per bin
}

df_agg = df.groupby(['Canteen_Section', 'time_bin']).agg(agg_dict).reset_index()

print(f"Aggregated shape: {df_agg.shape}")
print(f"Sections: {df_agg['Canteen_Section'].unique()}")
df_agg.head()

Aggregated shape: (74413, 5)
Sections: ['A' 'B' 'C' 'D']


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic
0,A,2015-01-01 12:00:00,2.98,5.96,120.0
1,A,2015-01-01 17:30:00,1.15,3.44,100.0
2,A,2015-01-01 18:00:00,0.93,2.78,100.0
3,A,2015-01-02 06:00:00,0.44,1.33,80.0
4,A,2015-01-02 12:00:00,1.54,4.62,120.0


## 3. Remove Zero‑Waste Bins (No Artificial Filling)

In [7]:
# Remove rows where no waste was recorded (Waste_Weight_kg == 0).
# This eliminates the large number of zero‑value entries that were previously
# added by the full reindexing step, keeping only bins that actually contain waste.
rows_before = len(df_agg)
df_agg = df_agg[df_agg['Waste_Weight_kg'] > 0].reset_index(drop=True)
rows_after = len(df_agg)
print(f"Rows before filter: {rows_before:,}")
print(f"Rows after filter : {rows_after:,} ({rows_before - rows_after:,} removed)")

# Note: The dataset now contains only time bins where waste was generated.
# There will be gaps in the time series, which the later feature engineering
# step (lags, rolling windows) will handle by shifting over the irregular sequence.
# This is intentional to reduce dataset size and avoid storing meaningless zeros.

df_agg.head()

Rows before filter: 74,413
Rows after filter : 74,413 (0 removed)


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic
0,A,2015-01-01 12:00:00,2.98,5.96,120.0
1,A,2015-01-01 17:30:00,1.15,3.44,100.0
2,A,2015-01-01 18:00:00,0.93,2.78,100.0
3,A,2015-01-02 06:00:00,0.44,1.33,80.0
4,A,2015-01-02 12:00:00,1.54,4.62,120.0


## 4. Add Time-Based & Cyclical Features

In [8]:
tb = df_agg['time_bin']

# Raw time features
df_agg['hour']        = tb.dt.hour
df_agg['minute']      = tb.dt.minute
df_agg['weekday']     = tb.dt.dayofweek          # 0=Monday, 6=Sunday
df_agg['month']       = tb.dt.month
df_agg['day_of_year'] = tb.dt.dayofyear
df_agg['quarter']     = tb.dt.quarter
df_agg['is_weekend']  = (tb.dt.dayofweek >= 5).astype(int)

# Cyclical encoding
df_agg['sin_hour']    = np.sin(2 * np.pi * df_agg['hour']    / 24)
df_agg['cos_hour']    = np.cos(2 * np.pi * df_agg['hour']    / 24)
df_agg['sin_minute']  = np.sin(2 * np.pi * df_agg['minute']  / 60)
df_agg['cos_minute']  = np.cos(2 * np.pi * df_agg['minute']  / 60)
df_agg['sin_weekday'] = np.sin(2 * np.pi * df_agg['weekday'] / 7)
df_agg['cos_weekday'] = np.cos(2 * np.pi * df_agg['weekday'] / 7)
df_agg['sin_month']   = np.sin(2 * np.pi * df_agg['month']   / 12)
df_agg['cos_month']   = np.cos(2 * np.pi * df_agg['month']   / 12)

print(f"Final shape: {df_agg.shape}")
print(f"Columns: {list(df_agg.columns)}")
df_agg.head()

Final shape: (74413, 20)
Columns: ['Canteen_Section', 'time_bin', 'Waste_Weight_kg', 'Cost_Loss', 'Foot_Traffic', 'hour', 'minute', 'weekday', 'month', 'day_of_year', 'quarter', 'is_weekend', 'sin_hour', 'cos_hour', 'sin_minute', 'cos_minute', 'sin_weekday', 'cos_weekday', 'sin_month', 'cos_month']


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic,hour,minute,weekday,month,day_of_year,quarter,is_weekend,sin_hour,cos_hour,sin_minute,cos_minute,sin_weekday,cos_weekday,sin_month,cos_month
0,A,2015-01-01 12:00:00,2.98,5.96,120.0,12,0,3,1,1,1,0,1.224647e-16,-1.000000e+00,0.000000e+00,1.0,0.433884,-0.900969,0.5,0.866025
1,A,2015-01-01 17:30:00,1.15,3.44,100.0,17,30,3,1,1,1,0,-9.659258e-01,-2.588190e-01,5.665539e-16,-1.0,0.433884,-0.900969,0.5,0.866025
2,A,2015-01-01 18:00:00,0.93,2.78,100.0,18,0,3,1,1,1,0,-1.000000e+00,-1.836970e-16,0.000000e+00,1.0,0.433884,-0.900969,0.5,0.866025
3,A,2015-01-02 06:00:00,0.44,1.33,80.0,6,0,4,1,2,1,0,1.000000e+00,6.123234e-17,0.000000e+00,1.0,-0.433884,-0.900969,0.5,0.866025
4,A,2015-01-02 12:00:00,1.54,4.62,120.0,12,0,4,1,2,1,0,1.224647e-16,-1.000000e+00,0.000000e+00,1.0,-0.433884,-0.900969,0.5,0.866025


## 5. Save Preprocessed Data

In [9]:
output_path = 'data/food_waste_preprocessed.csv'
df_agg.to_csv(output_path, index=False)
print(f"Saved to {output_path}  |  shape: {df_agg.shape}")

Saved to data/food_waste_preprocessed.csv  |  shape: (74413, 20)
